In [4]:
# ============================================================
# ERIP - SILVER LOAN DIMENSION
# Notebook      : nb_build_loan_dimension
# Layer         : Silver
#
# Purpose
# --------
# Build the enterprise Loan Dimension from the Bronze
# Loan Origination table.
#
# Business Objective
# ------------------
# Create a standardized, business-ready loan entity that
# will be consumed by:
#
# • Customer 360
# • Expected Credit Loss (ECL)
# • Basel III Reporting
# • IFRS 9 Analytics
# • Executive Dashboards
# • AI Decision Intelligence
#
# Enterprise Concepts
# -------------------
# ✓ Medallion Architecture
# ✓ Analytics Engineering
# ✓ Delta Lake
# ✓ Data Standardization
# ✓ Business Entity Modelling
# ✓ Enterprise Loan Dimension
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

# ------------------------------------------------------------
# SECTION 1 - Source / Target Configuration
# ------------------------------------------------------------

source_table = "bronze_loan_origination"
target_table = "silver_loan"
pipeline_name = "nb_build_loan_dimension"

# Pipeline Execution Timestamp
run_start_time = datetime.now()

print("ERIP Silver Loan Dimension Build Started")

StatementMeta(, 0fc3e3b5-e953-4851-af30-7330517e7d03, 6, Finished, Available, Finished, False)

ERIP Silver Loan Dimension Build Started


In [5]:
# ============================================================
# SECTION 2 - READ BRONZE LOAN TABLE
# ============================================================
#
# Purpose
# -------
# Read the validated/governed Bronze Delta table.
#
# Enterprise Concepts
# -------------------
# ✓ Delta Lake
# ✓ Data Lineage
# ✓ Medallion Architecture
# ============================================================

loan_bronze_df = spark.table(source_table)

display(loan_bronze_df.limit(10))

print(f"Rows read : {loan_bronze_df.count()}")
print(f"Columns   : {len(loan_bronze_df.columns)}")

StatementMeta(, 0fc3e3b5-e953-4851-af30-7330517e7d03, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2b199f8a-4827-4e6c-895f-bd6507a08923)

Rows read : 5000
Columns   : 26


In [6]:
# ============================================================
# SECTION 3 - STANDARDIZE LOAN DIMENSION
# ============================================================
#
# Purpose
# -------
# Standardize the Bronze Loan table into an enterprise-ready
# business entity.
#
# Standardization Activities
# --------------------------
# • Trim whitespace
# • Standardize text case
# • Convert numeric columns
# • Standardize dates
# • Remove duplicate Loan IDs
#
# Enterprise Concepts
# -------------------
# ✓ Data Standardization
# ✓ Business Entity
# ✓ Conformed Dimension
# ✓ Master Data Management
# ============================================================

silver_loan_df = (
    loan_bronze_df
    .select(
        upper(trim(col("loan_id"))).alias("loan_id"),
        upper(trim(col("facility_id"))).alias("facility_id"),
        upper(trim(col("customer_id"))).alias("customer_id"),
        upper(trim(col("customer_group_id"))).alias("customer_group_id"),
        initcap(trim(col("product_type"))).alias("product_type"),
        initcap(trim(col("facility_status"))).alias("facility_status"),
        to_date(col("origination_date")).alias("origination_date"),
        to_date(col("maturity_date")).alias("maturity_date"),
        col("approved_limit").cast("double").alias("approved_limit"),
        col("outstanding_balance").cast("double").alias("outstanding_balance"),
        col("undrawn_amount").cast("double").alias("undrawn_amount"),
        upper(trim(col("currency"))).alias("currency"),
        col("interest_rate_pct").cast("double").alias("interest_rate_pct"),
        initcap(trim(col("repayment_schedule"))).alias("repayment_schedule"),
        upper(trim(col("collateral_id"))).alias("collateral_id"),
        initcap(trim(col("collateral_type"))).alias("collateral_type"),
        col("collateral_value").cast("double").alias("collateral_value"),
        col("loan_to_value_pct").cast("double").alias("loan_to_value_pct"),
        initcap(trim(col("ifrs9_stage"))).alias("ifrs9_stage"),
        col("risk_weight").cast("double").alias("risk_weight"),
        col("risk_weighted_assets").cast("double").alias("risk_weighted_assets"),
        current_timestamp().alias("silver_updated_timestamp")
    )
    .dropDuplicates(["loan_id"])
)

StatementMeta(, 0fc3e3b5-e953-4851-af30-7330517e7d03, 8, Finished, Available, Finished, False)

In [7]:
# ============================================================
# SECTION 4 - BUSINESS ENRICHMENT
# ============================================================
#
# Purpose
# -------
# Create derived business attributes required for enterprise
# credit risk analytics.
#
# Derived Attributes
# ------------------
# • Loan Surrogate Key
# • Exposure at Default (EAD)
# • Loan Utilization
# • Loan Tenure
# • Remaining Maturity
# • Exposure Band
# • Secured / Unsecured Flag
# • IFRS 9 Numeric Stage
#
# Enterprise Concepts
# -------------------
# ✓ Basel III
# ✓ IFRS 9
# ✓ Exposure at Default
# ✓ Credit Risk Analytics
# ✓ Dimensional Modelling
# ✓ Customer 360 Foundation
# ============================================================

window_spec = Window.orderBy("loan_id")

silver_loan_df = (
    silver_loan_df
    .withColumn("loan_sk", row_number().over(window_spec))
    .withColumn(
        "exposure_at_default",
        col("outstanding_balance") +
        (col("undrawn_amount") * lit(0.45))
    )
    .withColumn(
        "utilization_pct",
        when(
            col("approved_limit") > 0,
            (col("outstanding_balance") /
             col("approved_limit")) * 100
        ).otherwise(0)
    )
    .withColumn(
        "loan_tenure_years",
        floor(
            months_between(
                col("maturity_date"),
                col("origination_date")
            ) / 12
        )
    )
    .withColumn(
        "remaining_maturity_years",
        floor(
            months_between(
                col("maturity_date"),
                current_date()
            ) / 12
        )
    )
    .withColumn(
        "exposure_band",
        when(col("exposure_at_default") >= 50000000, "Very Large Exposure")
        .when(col("exposure_at_default") >= 10000000, "Large Exposure")
        .when(col("exposure_at_default") >= 1000000, "Medium Exposure")
        .otherwise("Small Exposure")
    )
    .withColumn(
        "secured_flag",
        when(
            col("collateral_type") == "Unsecured",
            "N"
        ).otherwise("Y")
    )
    .withColumn(
        "ifrs9_stage_numeric",
        regexp_extract(
            col("ifrs9_stage"),
            r"(\d+)",
            1
        ).cast("int")
    )

    .select(
        "loan_sk",
        "loan_id",
        "facility_id",
        "customer_id",
        "customer_group_id",
        "product_type",
        "facility_status",
        "origination_date",
        "maturity_date",
        "loan_tenure_years",
        "remaining_maturity_years",
        "approved_limit",
        "outstanding_balance",
        "undrawn_amount",
        "exposure_at_default",
        "utilization_pct",
        "exposure_band",
        "currency",
        "interest_rate_pct",
        "repayment_schedule",
        "secured_flag",
        "collateral_id",
        "collateral_type",
        "collateral_value",
        "loan_to_value_pct",
        "ifrs9_stage",
        "ifrs9_stage_numeric",
        "risk_weight",
        "risk_weighted_assets",
        "silver_updated_timestamp"
    )
)

display(silver_loan_df.limit(10))

StatementMeta(, 0fc3e3b5-e953-4851-af30-7330517e7d03, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 63a261c8-74e0-4a13-a634-ce2dce478451)

In [8]:
# ============================================================
# SECTION 5 - SILVER DATA QUALITY VALIDATION
# ============================================================
#
# Purpose
# -------
# Validate the Silver Loan Dimension before publishing.
#
# Validation Checks
# -----------------
# • Duplicate Loan IDs
# • Null Loan IDs
# • Null Surrogate Keys
# • Negative Exposure
# • Invalid Utilization
#
# Enterprise Concepts
# -------------------
# ✓ Data Quality
# ✓ Data Governance
# ✓ Trusted Analytics Layer
# ============================================================

total_rows = silver_loan_df.count()

duplicate_loan_ids = total_rows - silver_loan_df.select("loan_id").distinct().count()

null_loan_ids = silver_loan_df.filter(
    col("loan_id").isNull()
).count()

null_surrogate_keys = silver_loan_df.filter(
    col("loan_sk").isNull()
).count()

invalid_exposure = silver_loan_df.filter(
    col("exposure_at_default") < 0
).count()

invalid_utilization = silver_loan_df.filter(
    col("utilization_pct") < 0
).count()

print("Silver Loan Quality Checks")
print("--------------------------")
print(f"Rows: {total_rows}")
print(f"Duplicate Loan IDs: {duplicate_loan_ids}")
print(f"Null Loan IDs: {null_loan_ids}")
print(f"Null Surrogate Keys: {null_surrogate_keys}")
print(f"Invalid Exposure: {invalid_exposure}")
print(f"Invalid Utilization: {invalid_utilization}")

if (
    duplicate_loan_ids > 0 or
    null_loan_ids > 0 or
    null_surrogate_keys > 0 or
    invalid_exposure > 0 or
    invalid_utilization > 0
):
    raise Exception("Silver Loan Validation Failed")
else:
    print("✓ Silver Loan Validation Passed")

StatementMeta(, 0fc3e3b5-e953-4851-af30-7330517e7d03, 10, Finished, Available, Finished, False)

Silver Loan Quality Checks
--------------------------
Rows: 5000
Duplicate Loan IDs: 0
Null Loan IDs: 0
Null Surrogate Keys: 0
Invalid Exposure: 0
Invalid Utilization: 0
✓ Silver Loan Validation Passed


In [9]:
# ============================================================
# SECTION 6 - WRITE SILVER DELTA TABLE
# ============================================================
#
# Purpose
# -------
# Publish the enterprise Loan Dimension to the Silver Layer.
#
# Output
# ------
# silver_loan
#
# Enterprise Concepts
# -------------------
# ✓ Delta Lake
# ✓ Analytics Engineering
# ✓ Enterprise Data Platform
# ============================================================

silver_loan_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(target_table)

print(f"✓ Silver table created: {target_table}")
print(f"Rows written: {silver_loan_df.count()}")

StatementMeta(, 0fc3e3b5-e953-4851-af30-7330517e7d03, 11, Finished, Available, Finished, False)

✓ Silver table created: silver_loan
Rows written: 5000
